In [ ]:
# --- JASON DEVELOPMENT 93.3 ---
# Endringer fra 93.2:
# - ICT/90 fargeskala endret:
#   lav verdi = lys gul
#   høy verdi = dus lilla
# - Øvrig struktur beholdt fra 93.2

import requests
import pandas as pd
import numpy as np
from google.colab import auth
import gspread
from google.auth import default
import urllib3
import math
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open("Jason Development")


def clean_float(val):
    try:
        return float(val)
    except:
        return 0.0


def get_fdr_multiplier_v87(fdr_score):
    f = round(fdr_score)
    return (
        1.28 if f <= 1 else
        1.20 if f == 2 else
        1.00 if f == 3 else
        0.88 if f == 4 else
        0.82
    )


def calculate_progressive_xp_v87(ps_val, avg_min):
    if avg_min < 30 or ps_val <= 0:
        return 0.0
    return round(max(0.0, (ps_val * 0.10)), 1)


def trend_symbol(diff, threshold):
    if diff > threshold:
        return "📈"
    elif diff < -threshold:
        return "📉"
    else:
        return "➖"


def level_symbol(val):
    if val >= 0.80:
        return "🟢"
    elif val >= 0.45:
        return "🟡"
    else:
        return "🔴"


def combined_trend_symbol(last6, season, threshold):
    level = level_symbol(season)
    diff = last6 - season
    trend = trend_symbol(diff, threshold)
    return f"{level}{trend}"


def reset_conditional_formatting(ws):
    try:
        metadata = sh.fetch_sheet_metadata()
        rule_count = 0

        for sheet in metadata.get("sheets", []):
            if sheet["properties"]["sheetId"] == ws.id:
                rule_count = len(sheet.get("conditionalFormats", []))
                break

        requests_list = []
        for _ in range(rule_count):
            requests_list.append({
                "deleteConditionalFormatRule": {
                    "sheetId": ws.id,
                    "index": 0
                }
            })

        if requests_list:
            sh.batch_update({"requests": requests_list})
    except:
        pass


def update_worksheet(sheet_name, dataframe):
    print(f"Oppdaterer {sheet_name}...")

    df_clean = dataframe.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows="2000",
            cols="120"
        )

    ws.update(
        [df_clean.columns.tolist()] +
        df_clean.values.tolist()
    )

    time.sleep(5)


def get_or_create_worksheet(sheet_name, rows="2000", cols="120"):
    try:
        ws = sh.worksheet(sheet_name)
    except:
        ws = sh.add_worksheet(
            title=sheet_name,
            rows=rows,
            cols=cols
        )
    return ws


def col_to_a1(col_num):
    result = ""
    while col_num:
        col_num, remainder = divmod(col_num - 1, 26)
        result = chr(65 + remainder) + result
    return result


def apply_observer_formatting(ws, df, data_ranges):
    reset_conditional_formatting(ws)

    columns = df.columns.tolist()
    requests_list = []

    column_widths = {
        "Name": 180, "Club": 55, "Pos": 55, "Price": 65, "xMin": 65,
        "xP": 65, "PS(5)": 70,
        "PS1": 60, "PS2": 60, "PS3": 60, "PS4": 60, "PS5": 60,
        "Heat": 75, "GIΔ": 60, "xGIΔ": 65,
        "ICT/90": 70, "ICTΔ": 60,
        "Form": 65, "PPM LAST 6": 90, "P6": 55, "PPM": 60,
        "BxGI/90": 75,
        "Mins guide": 95, "Min 6": 65,
        "GI": 50, "xGI": 60,
        "GI (6)": 65, "xGI (6)": 70,
        "Pot": 55,
        "GI/90 (6)": 75, "xGI/90 (6)": 85,
        "GI/90 (S)": 75, "xGI/90 (S)": 85
    }

    for idx, col in enumerate(columns):
        requests_list.append({
            "updateDimensionProperties": {
                "range": {
                    "sheetId": ws.id,
                    "dimension": "COLUMNS",
                    "startIndex": idx,
                    "endIndex": idx + 1
                },
                "properties": {"pixelSize": column_widths.get(col, 70)},
                "fields": "pixelSize"
            }
        })

    for block in data_ranges:
        header_row_index = block["header_row"] - 1

        requests_list.append({
            "repeatCell": {
                "range": {
                    "sheetId": ws.id,
                    "startRowIndex": header_row_index,
                    "endRowIndex": header_row_index + 1,
                    "startColumnIndex": 0,
                    "endColumnIndex": len(columns)
                },
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": {
                            "red": 1.0,
                            "green": 1.0,
                            "blue": 1.0
                        },
                        "textFormat": {
                            "bold": True,
                            "foregroundColor": {
                                "red": 0.0,
                                "green": 0.0,
                                "blue": 0.0
                            }
                        },
                        "horizontalAlignment": "CENTER"
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,textFormat,horizontalAlignment)"
            }
        })

    # Blå: lav = hvit, høy = #337fe5
    blue_white_cols = [
        "xP",
        "Form",
        "P6",
        "BxGI/90",
        "GI (6)",
        "GI/90 (6)",
        "GI/90 (S)"
    ]

    for col_name in blue_white_cols:
        if col_name in columns:
            c_idx = columns.index(col_name)

            for block in data_ranges:
                if block["num_rows"] <= 0:
                    continue

                requests_list.append({
                    "addConditionalFormatRule": {
                        "rule": {
                            "ranges": [{
                                "sheetId": ws.id,
                                "startRowIndex": block["data_start"] - 1,
                                "endRowIndex": block["data_end"],
                                "startColumnIndex": c_idx,
                                "endColumnIndex": c_idx + 1
                            }],
                            "gradientRule": {
                                "minpoint": {
                                    "color": {
                                        "red": 1.0,
                                        "green": 1.0,
                                        "blue": 1.0
                                    },
                                    "type": "MIN"
                                },
                                "maxpoint": {
                                    "color": {
                                        "red": 0.2,
                                        "green": 0.498,
                                        "blue": 0.898
                                    },
                                    "type": "MAX"
                                }
                            }
                        },
                        "index": 0
                    }
                })

    # PS: lav = lys gul 2, høy = #57bb8a
    ps_cols = ["PS(5)", "PS1", "PS2", "PS3", "PS4", "PS5"]

    for col_name in ps_cols:
        if col_name in columns:
            c_idx = columns.index(col_name)

            for block in data_ranges:
                if block["num_rows"] <= 0:
                    continue

                requests_list.append({
                    "addConditionalFormatRule": {
                        "rule": {
                            "ranges": [{
                                "sheetId": ws.id,
                                "startRowIndex": block["data_start"] - 1,
                                "endRowIndex": block["data_end"],
                                "startColumnIndex": c_idx,
                                "endColumnIndex": c_idx + 1
                            }],
                            "gradientRule": {
                                "minpoint": {
                                    "color": {
                                        "red": 1.0,
                                        "green": 0.949,
                                        "blue": 0.800
                                    },
                                    "type": "MIN"
                                },
                                "maxpoint": {
                                    "color": {
                                        "red": 0.341,
                                        "green": 0.733,
                                        "blue": 0.541
                                    },
                                    "type": "MAX"
                                }
                            }
                        },
                        "index": 0
                    }
                })

    # ICT/90: lav = lys gul, høy = dus lilla
    if "ICT/90" in columns:
        c_idx = columns.index("ICT/90")

        for block in data_ranges:
            if block["num_rows"] <= 0:
                continue

            requests_list.append({
                "addConditionalFormatRule": {
                    "rule": {
                        "ranges": [{
                            "sheetId": ws.id,
                            "startRowIndex": block["data_start"] - 1,
                            "endRowIndex": block["data_end"],
                            "startColumnIndex": c_idx,
                            "endColumnIndex": c_idx + 1
                        }],
                        "gradientRule": {
                            "minpoint": {
                                "color": {
                                    "red": 1.0,
                                    "green": 0.949,
                                    "blue": 0.800
                                },
                                "type": "MIN"
                            },
                            "maxpoint": {
                                "color": {
                                    "red": 0.710,
                                    "green": 0.494,
                                    "blue": 0.863
                                },
                                "type": "MAX"
                            }
                        }
                    },
                    "index": 0
                }
            })

    if requests_list:
        sh.batch_update({"requests": requests_list})


def update_observer_worksheet(dataframe):
    print("Oppdaterer The Observer...")

    start_row = 16

    df_clean = dataframe.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0)

    ws = get_or_create_worksheet(
        "The Observer",
        rows="2000",
        cols="80"
    )

    last_col_letter = col_to_a1(len(df_clean.columns))
    clear_range = f"A{start_row}:{last_col_letter}2000"
    ws.batch_clear([clear_range])

    blocks = [
        ("DEF", 21),
        ("MID", 31),
        ("FWD", 21),
        ("GKP", 11)
    ]

    all_values = []
    data_ranges = []
    current_row = start_row

    for pos, limit in blocks:
        block_df = (
            df_clean[df_clean["Pos"] == pos]
            .sort_values("xP", ascending=False)
            .head(limit)
        )

        all_values.append(df_clean.columns.tolist())

        header_row = current_row
        data_start = current_row + 1

        rows = block_df.values.tolist()

        for row in rows:
            all_values.append(row)

        data_end = data_start + len(rows) - 1

        data_ranges.append({
            "pos": pos,
            "header_row": header_row,
            "data_start": data_start,
            "data_end": data_end,
            "num_rows": len(rows)
        })

        current_row = data_end + 2
        all_values.append([""] * len(df_clean.columns))

    ws.update(
        f"A{start_row}",
        all_values
    )

    apply_observer_formatting(
        ws,
        df_clean,
        data_ranges
    )

    time.sleep(5)


def run_jason_development_93_3():
    base_url = "https://fantasy.premierleague.com/api/"

    r = requests.get(
        f"{base_url}bootstrap-static/",
        verify=False
    ).json()

    elements = pd.DataFrame(r['elements'])

    try:
        next_gw = next(
            event['id']
            for event in r['events']
            if event['is_next']
        )
    except StopIteration:
        next_gw = 1

    gw_cols = [
        f"PS{next_gw}",
        f"PS{next_gw+1}",
        f"PS{next_gw+2}",
        f"PS{next_gw+3}",
        f"PS{next_gw+4}"
    ]

    xp_cols = [
        f"xP{next_gw}",
        f"xP{next_gw+1}",
        f"xP{next_gw+2}",
        f"xP{next_gw+3}",
        f"xP{next_gw+4}"
    ]

    target_gws = list(range(next_gw, next_gw + 5))

    teams_det = {
        t['id']: {
            'short': t['short_name'],
            'cs_factor': (
                t['strength_defence_home'] +
                t['strength_defence_away']
            ) / 6000
        }
        for t in r['teams']
    }

    fixtures_r = requests.get(
        f"{base_url}fixtures/",
        verify=False
    ).json()

    last_6_gws = list(range(max(1, next_gw - 6), next_gw))

    main_list = []
    observer_list = []
    season_archive = []
    team_archive = []

    for t in r['teams']:
        team_archive.append({
            'Team': t['name'],
            'Short': t['short_name'],
            'Strength Overall Home': t['strength_overall_home'],
            'Strength Overall Away': t['strength_overall_away'],
            'Strength Attack Home': t['strength_attack_home'],
            'Strength Attack Away': t['strength_attack_away'],
            'Strength Defence Home': t['strength_defence_home'],
            'Strength Defence Away': t['strength_defence_away']
        })

    for _, p in elements.iterrows():
        p_id = p['id']
        team_id = p['team']
        pos_id = p['element_type']

        is_injured = p['status'] in ['d', 'i', 's', 'u']

        try:
            summary = requests.get(
                f"{base_url}element-summary/{p_id}/",
                verify=False
            ).json()

            hist_full = pd.DataFrame(
                summary.get('history', [])
            )

        except:
            continue

        if hist_full.empty:
            continue

        hist_s6 = hist_full[
            hist_full['round'].isin(last_6_gws)
        ]

        m6 = hist_s6['minutes'].sum()

        x6 = round(
            hist_s6['expected_goal_involvements']
            .apply(clean_float)
            .sum(),
            2
        )

        g6 = int(
            hist_s6['goals_scored'].sum() +
            hist_s6['assists'].sum()
        )

        ict6 = round(
            hist_s6['ict_index']
            .apply(clean_float)
            .sum(),
            2
        )

        hist_agg = hist_s6.groupby('round').agg({
            'total_points': 'sum',
            'minutes': 'sum',
            'element': 'count'
        }).to_dict('index')

        f_guide = ""
        min_g = ""
        t_sum = 0

        for gw in last_6_gws:
            if any(
                f['event'] == gw and
                (
                    f['team_h'] == team_id or
                    f['team_a'] == team_id
                )
                for f in fixtures_r
            ):
                d = hist_agg.get(gw)

                if d:
                    t_sum += math.sqrt(
                        max(0, d['total_points'])
                    )

                    num_matches = d.get('element', 1)

                    green_limit = 6 * num_matches
                    yellow_limit = 4 * num_matches

                    f_guide += (
                        "🟢"
                        if d['total_points'] >= green_limit else
                        "🟡"
                        if d['total_points'] >= yellow_limit else
                        "🔴"
                        if d['minutes'] > 0 else
                        "⚪"
                    )

                    min_g += (
                        "🟢"
                        if d['minutes'] >= (78 * num_matches) else
                        "🟡"
                        if d['minutes'] >= (60 * num_matches) else
                        "🔴"
                        if d['minutes'] > 0 else
                        "⚪"
                    )
                else:
                    f_guide += "⚪"
                    min_g += "⚪"
            else:
                f_guide += "X"
                min_g += "X"

        v_form = round((t_sum / 6) * 4, 1)

        team_f_6 = [
            f for f in fixtures_r
            if f['event'] in last_6_gws and
            (
                f['team_h'] == team_id or
                f['team_a'] == team_id
            )
        ]

        avg_min_6 = (
            min(90.0, m6 / len(team_f_6))
            if len(team_f_6) > 0 else 0
        )

        p6_points = int(hist_s6['total_points'].sum())

        ppm_last_6 = (
            round(p6_points / len(team_f_6), 2)
            if len(team_f_6) > 0 else 0.0
        )

        gi_90_6 = (
            round((g6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        xgi_90_6 = (
            round((x6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        season_minutes = (
            float(p['minutes'])
            if float(p['minutes']) > 0 else 1.0
        )

        season_gi = int(
            p['goals_scored'] +
            p['assists']
        )

        season_xgi = round(
            clean_float(
                p['expected_goal_involvements']
            ),
            2
        )

        season_ict = round(
            clean_float(p['ict_index']),
            2
        )

        season_gi_90 = round(
            (season_gi / season_minutes) * 90,
            2
        )

        season_xgi_90 = round(
            (season_xgi / season_minutes) * 90,
            2
        )

        season_ict_90 = round(
            (season_ict / season_minutes) * 90,
            2
        )

        gi_trend = combined_trend_symbol(
            gi_90_6,
            season_gi_90,
            0.15
        )

        xgi_trend = combined_trend_symbol(
            xgi_90_6,
            season_xgi_90,
            0.15
        )

        ict_90_6 = (
            round((ict6 / m6 * 90), 2)
            if m6 > 0 else 0.0
        )

        ict_trend = trend_symbol(
            ict_90_6 - season_ict_90,
            1.5
        )

        heat_score = round(
            (
                (gi_90_6 - season_gi_90) +
                (xgi_90_6 - season_xgi_90)
            ) / 2,
            2
        )

        if avg_min_6 < 30:
            heat_symbol = "---"
        else:
            if heat_score >= 0.55:
                heat_symbol = "🔥🔥🔥"
            elif heat_score >= 0.30:
                heat_symbol = "🔥🔥"
            elif heat_score >= 0.10:
                heat_symbol = "🔥"
            elif heat_score > -0.10:
                heat_symbol = "➖"
            elif heat_score > -0.30:
                heat_symbol = "💩"
            else:
                heat_symbol = "💩💩"

        xgi_bal = (
            (xgi_90_6 * 0.3) +
            (
                (
                    season_xgi /
                    season_minutes *
                    90
                ) * 0.7
            )
        )

        base_p = (
            (
                float(p['points_per_game']) *
                10.0 *
                0.60
            ) +
            (
                (
                    (xgi_bal * 45.0) +
                    (v_form * 2.8) +
                    (
                        (
                            p['bps'] /
                            season_minutes *
                            90
                        ) / 5
                    )
                ) * 0.40
            )
        )

        if pos_id == 2:
            base_p += (
                teams_det[team_id]['cs_factor'] *
                22.0
            )

        if (
            (
                p['total_points'] >= 25 or
                next_gw < 20
            ) and
            (m6 >= 90 or next_gw == 1)
        ):
            adj_ps = []

            for gw in target_gws:
                gw_f = [
                    f for f in fixtures_r
                    if f['event'] == gw and
                    (
                        f['team_h'] == team_id or
                        f['team_a'] == team_id
                    )
                ]

                adj_ps.append(
                    sum(
                        base_p *
                        get_fdr_multiplier_v87(
                            float(
                                f['team_h_difficulty']
                                if f['team_h'] == team_id
                                else f['team_a_difficulty']
                            )
                        )
                        for f in gw_f
                    )
                )

            p_dict = {
                'Navn': f"{'⚠️ ' if is_injured else ''}{p['first_name']} {p['second_name']}",
                'Lag': teams_det[team_id]['short'],
                'Pos': {
                    1: 'GKP',
                    2: 'DEF',
                    3: 'MID',
                    4: 'FWD'
                }[pos_id],
                'Pris': float(p['now_cost']) / 10,
                'Eie %': p['selected_by_percent'],
                'Form': v_form,
                'Siste 6': f_guide,
                'PPK': float(p['points_per_game']),
                'BPS/90': round(
                    (
                        p['bps'] /
                        season_minutes *
                        90
                    ),
                    1
                ),
                'BxGI/90': round(xgi_bal, 2),
                'PS (N5)': round(sum(adj_ps) / 5, 1),
                'PS Tot': round(sum(adj_ps), 1)
            }

            for i, name in enumerate(gw_cols):
                p_dict[name] = round(adj_ps[i], 1)

            txp = []

            for i, ps_v in enumerate(adj_ps):
                v = calculate_progressive_xp_v87(
                    ps_v,
                    avg_min_6
                )
                p_dict[xp_cols[i]] = v
                txp.append(v)

            p_dict.update({
                'SUM5': round(sum(txp), 1),
                'P6': p6_points,
                'GI': season_gi,
                'xGI': season_xgi,
                'xGI (S6)': x6,
                'GI (S6)': g6,
                'Pot (6)': round(x6 - g6, 2),
                'GI/90 (6)': gi_90_6,
                'xGI/90 (6)': xgi_90_6,
                'GI/90 (S)': season_gi_90,
                'xGI/90 (S)': season_xgi_90,
                'HeatScore': heat_score,
                'Heat': heat_symbol,
                'Merknad': (
                    p['news']
                    if is_injured else ""
                ),
                'Total min sesong': p['minutes'],
                'Min S/6': round(avg_min_6, 1),
                'Min-guide': min_g,
                'Status': "---",
                'GIΔ': gi_trend,
                'xGIΔ': xgi_trend,
                'ICT/90': season_ict_90,
                'ICTΔ': ict_trend
            })

            main_list.append(p_dict)

            observer_list.append({
                "Name": p_dict["Navn"],
                "Club": p_dict["Lag"],
                "Pos": p_dict["Pos"],
                "Price": p_dict["Pris"],
                "xMin": round(avg_min_6, 1),
                "xP": txp[0] if len(txp) > 0 else 0.0,
                "PS(5)": p_dict["PS (N5)"],
                "PS1": round(adj_ps[0], 1) if len(adj_ps) > 0 else 0.0,
                "PS2": round(adj_ps[1], 1) if len(adj_ps) > 1 else 0.0,
                "PS3": round(adj_ps[2], 1) if len(adj_ps) > 2 else 0.0,
                "PS4": round(adj_ps[3], 1) if len(adj_ps) > 3 else 0.0,
                "PS5": round(adj_ps[4], 1) if len(adj_ps) > 4 else 0.0,
                "Heat": heat_symbol,
                "GIΔ": gi_trend,
                "xGIΔ": xgi_trend,
                "ICT/90": season_ict_90,
                "ICTΔ": ict_trend,
                "Form": v_form,
                "PPM LAST 6": ppm_last_6,
                "P6": p6_points,
                "PPM": float(p['points_per_game']),
                "BxGI/90": round(xgi_bal, 2),
                "Mins guide": min_g,
                "Min 6": round(avg_min_6, 1),
                "GI": season_gi,
                "xGI": season_xgi,
                "GI (6)": g6,
                "xGI (6)": x6,
                "Pot": round(x6 - g6, 2),
                "GI/90 (6)": gi_90_6,
                "xGI/90 (6)": xgi_90_6,
                "GI/90 (S)": season_gi_90,
                "xGI/90 (S)": season_xgi_90
            })

        season_archive.append({
            'Player ID': p_id,
            'Navn': f"{p['first_name']} {p['second_name']}",
            'Lag': teams_det[team_id]['short'],
            'Pos': {
                1: 'GKP',
                2: 'DEF',
                3: 'MID',
                4: 'FWD'
            }[pos_id],
            'Pris': float(p['now_cost']) / 10,
            'Minutes': p['minutes'],
            'Points': p['total_points'],
            'PPG': p['points_per_game'],
            'GI': season_gi,
            'xGI': season_xgi,
            'GI/90': season_gi_90,
            'xGI/90': season_xgi_90,
            'ICT/90': season_ict_90,
            'BPS': p['bps'],
            'Selected %': p['selected_by_percent']
        })

    df_f = pd.DataFrame(main_list)

    if df_f.empty:
        print("Ingen spillere funnet.")
        return

    order = [
        'Navn',
        'Lag',
        'Pos',
        'Pris',
        'Eie %',
        'Form',
        'Siste 6',
        'PPK',
        'BPS/90',
        'BxGI/90',
        'PS (N5)',
        'PS Tot'
    ] + gw_cols + xp_cols + [
        'SUM5',
        'P6',
        'GI',
        'xGI',
        'xGI (S6)',
        'GI (S6)',
        'Pot (6)',
        'GI/90 (6)',
        'xGI/90 (6)',
        'GI/90 (S)',
        'xGI/90 (S)',
        'HeatScore',
        'Heat',
        'Merknad',
        'Total min sesong',
        'Min S/6',
        'Min-guide',
        'Status',
        'GIΔ',
        'xGIΔ',
        'ICT/90',
        'ICTΔ'
    ]

    update_worksheet(
        "ICT",
        df_f.sort_values(
            'PS Tot',
            ascending=False
        )[order]
    )

    for s, pos in {
        "Keeper": "GKP",
        "Forsvar": "DEF",
        "Midtbane": "MID",
        "Angrep": "FWD"
    }.items():
        update_worksheet(
            s,
            df_f[df_f['Pos'] == pos]
            .sort_values(
                'PS Tot',
                ascending=False
            )[order]
        )

    update_worksheet(
        "xP",
        df_f.sort_values(
            'SUM5',
            ascending=False
        )[order]
    )

    observer_order = [
        "Name",
        "Club",
        "Pos",
        "Price",
        "xMin",
        "xP",
        "PS(5)",
        "PS1",
        "PS2",
        "PS3",
        "PS4",
        "PS5",
        "Heat",
        "GIΔ",
        "xGIΔ",
        "ICT/90",
        "ICTΔ",
        "Form",
        "PPM LAST 6",
        "P6",
        "PPM",
        "BxGI/90",
        "Mins guide",
        "Min 6",
        "GI",
        "xGI",
        "GI (6)",
        "xGI (6)",
        "Pot",
        "GI/90 (6)",
        "xGI/90 (6)",
        "GI/90 (S)",
        "xGI/90 (S)"
    ]

    df_observer = pd.DataFrame(observer_list)

    update_observer_worksheet(
        df_observer[observer_order]
    )

    update_worksheet(
        "Season_Data",
        pd.DataFrame(season_archive)
    )

    update_worksheet(
        "Team_Data",
        pd.DataFrame(team_archive)
    )

    print(
        "Jason Development 93.3 ferdig "
        "med endret ICT/90-fargeskala."
    )


run_jason_development_93_3()

Oppdaterer ICT...
Oppdaterer Keeper...
Oppdaterer Forsvar...
Oppdaterer Midtbane...
Oppdaterer Angrep...
Oppdaterer xP...
Oppdaterer The Observer...


/tmp/ipykernel_9063/2032505631.py:418: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(


Oppdaterer Season_Data...
Oppdaterer Team_Data...
Jason Development 93.3 ferdig med endret ICT/90-fargeskala.
